# CDP_FACT_SALES Exploration


## 1. Objective
The purpose of this analysis is to

### 1. Identify the true Primary Key (PK) for the CDP_FACT_SALES table
- What happens when the PK is NULL?
### 2. Understand Customer Key relationships
- Difference between SPCustomerKey and ATCustomerKey
- Are they unique? Are they tied together?
### 3. Determine which key should be used for marketing

## 2. Key Findings (Executive Summary)
### Primary Key Behavior

- GlobalId behaves like a Primary Key for reportable transactions
- However, a large number of rows (~288K in 2026) have NULL GlobalId
- These NULLs are mostly tied to:
    - Shipping & Service Fees
    - Non-reportable transactions
    - Refunds

### Conclusion:
GlobalId is **not a universal PK** — it is conditional based on transaction type.

### Customer Key Logic

- SPCustomerKey → Pass Owner (WHO owns the pass)
- ATCustomerKey → Purchaser (WHO bought the pass)

**These are NOT always the same person**
#### Relationship:

- GlobalId → SPCustomerKey = one-to-many
- SPCustomerKey → GlobalId = one-to-one
- ATCustomerKey is less directly tied GlobalId but to SPCustomer 

### Marketing Implication

SPCustomerKey = best for lifecycle / loyalty targeting\
ATCustomerKey = useful for purchaser behavior

## Recommended:
Use SPCustomerKey as primary marketing key, with fallback logic to ATCustomerKey when missing

## 3.Analysis

## 1. What is the Primary Key(PK)?

I first did a count on all null found in the sales related to the SPkey, ATkey, and GlobalId. Since all properties have started during ticket year 2026 i made my focus of this ticket year.

In [ ]:
%%sql -r dataframe_1
SELECT 
SUM(CASE WHEN "SPCustomerKey" LIKE '%None%' THEN 1 ELSE 0 END) AS SPCustomerkey,
SUM(CASE WHEN "ATCustomerKey" LIKE '%None%' THEN 1 ELSE 0 END) AS ATCustomerkey,
SUM(CASE WHEN "GlobalId" IS NULL THEN 1 ELSE 0 END) AS GlobalID

FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES 
WHERE "SeasonKey" LIKE '%2026%'


### Insight
- GlobalId have the least null values by a wide margin.


Now we will see the comparison at the between each of the three 

In [ ]:
%%sql -r dataframe_2
-- removing "None" since those are null values
WITH CTE AS (
SELECT DISTINCT
"ATCustomerKey",
count(DISTINCT "GlobalId") as SPkey,
count(DISTINCT "SPCustomerKey") as ATkey,
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES 
WHERE 1=1
AND "SeasonKey" LIKE '%2026%'
AND 
(
"GlobalId" IS NOT NULL
AND "ATCustomerKey" NOT LIKE '%None%'
AND "SPCustomerKey" NOT LIKE '%None%'
)
GROUP BY ALL
)
SELECT
ATkey as keis,
count(*)
FROM CTE
GROUP BY ALL 
ORDER BY keis


### GlobalID as Anchor
Here we can see that there is a many to one relationship from the GlobalId being the anchor.\
These two tables can tell us that the SPCustomerKey and the GlobalId are more 1:1 than the ATCustomerKey

---


|SPKeys under one GlobalId|	COUNT(*)|
|-|-|
|1|	4,936,327|
|2|	2,774|
|3|	15|
|4|	1|
----------


|ATkes under one GlobalId|	COUNT(*)|
|-|-|
|1|	4,757,644
|2|	149,328
|3|	30,576
|4|	1,492
|5|	69
|6|	8





### SPkey as Anchor
- SPkey as the anchor we see a direct 1:1 connection to the GlobalIds

|GlobalIds under on SPkey| COUNT(*)|
|-|-|
|1|4,941,924|
---
|ATkeys under one SPkey|	COUNT(*)|
|-|-|
|1|	4,762,586
|2|	147,783
|3|	30,055
|4|	1,435
|5|	58
|6|	7

### GlobalIds -> SPKey = 1:Many **BUT**  SPkey -> GlobalIds = 1:1? 

In [ ]:
%%sql -r dataframe_3
SELECT a."GlobalId",a."Barcode",a."SPCustomerKey",a."SiteKey"
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_SPCUSTOMERS a 
JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES b 
    ON a."SPCustomerKey" = b."SPCustomerKey"
WHERE 1=1
AND "SeasonKey" LIKE '%2026%'
AND a."GlobalId" = 'e3a8e9dec148ee118152005056aefa59TREX'

**To Note**: While looking at the DIM_SP/AT Customer Tables, AT does not have a GlobalId assigned to the ATKey while SPkey does in its own table.
 After Grabbing an individual with one GlobalId but multiple SPKeys, we can see that SPkeys are made by park (from what I understand each park is siloed into its own and then merged together by IT).
 ### For that reason we see why a GlobalId would have more than 1 SPkey. 

<div style="display:flex; align-items:center; gap:20px;">

  <!-- Main -->
  <input 
    input placeholder='GlobalID'
    style="padding:10px; border:2px solid green; border-radius:8px;"
  />

  <!-- Arrow (optional visual) -->
  <div style="font-size:24px;">→</div>

  <!-- Children -->
  <div style="display:flex; flex-direction:column; gap:10px;">
    <input placeholder="CW|XXXX" style="padding:10px; border:2px solid blue; border-radius:8px;">
    <input placeholder="CP|XXXX" style="padding:10px; border:2px solid blue; border-radius:8px;">
    <input placeholder="OG|XXXX" style="padding:10px; border:2px solid blue; border-radius:8px;">
  </div>

</div>

## GlobalIDs that are null?

In [ ]:
%%sql -r dataframe_4
SELECT 
    b."AttendanceGroupCategoryDescription",
    b."AttendanceGroupSubCategoryDescription",
    b."AttendanceGroupSubCategoryName",
    b."ReportableSP"
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES a 
JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_PRODUCTS b 
    ON a."ProductKey" = b."ProductKey"
WHERE 1=1
AND "SeasonKey" LIKE '%2026%'
AND "GlobalId" IS NULL 
--total rows: 288,879

In [ ]:
for col in dataframe_4.columns:
    print(f"\nColumn: {col}")
    print(dataframe_4[col].value_counts(dropna=False))

While looking at all of the (AttendanceGroupSubCategory)AGSC Descriptions we can see that a large number of null GlobalIds are from Shipping/Service Fees & MISC Fast Lane?\
There is a column in DIM Products that has ReportableSP, this matches 99% of Null GlobalIds



In [ ]:
%%sql -r dataframe_5
SELECT b."AttendanceGroupCategoryDescription",
b."AttendanceGroupSubCategoryDescription",
b."AttendanceGroupSubCategoryName",
a."SalesQuantity",
a."SalesAmount",
split_part("EventTypeKey",'|',2) as eventtype,
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES a 
JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_PRODUCTS b 
    ON a."ProductKey" = b."ProductKey"
WHERE 1=1
AND "SeasonKey" LIKE '%2026%'
AND "GlobalId" IS NULL 
AND "ReportableSP" = 'Reportable SP Sales'

In [ ]:
for col in dataframe_5.columns:
    print(f"\nColumn: {col}")
    print(dataframe_5[col].value_counts(dropna=False))

In [ ]:
%%sql -r dataframe_6
SELECT b."AttendanceGroupCategoryDescription",
b."AttendanceGroupSubCategoryDescription",
b."AttendanceGroupSubCategoryName",
b."ReportableSP",
a."SalesQuantity",
a."SalesAmount",
split_part("EventTypeKey",'|',2) as eventtype,
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES a 
JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_PRODUCTS b 
    ON a."ProductKey" = b."ProductKey"
WHERE 1=1
AND "SeasonKey" LIKE '%2026%'
AND "GlobalId" IS NULL 
AND "ReportableSP" = 'Reportable SP Sales'
AND( "EventTypeKey" NOT LIKE '%NOOP%')

In [ ]:
for col in dataframe_6.columns:
    print(f"\nColumn: {col}")
    print(dataframe_6[col].value_counts(dropna=False))

Looking at the 1% that is Reportable SP Sales those are where the eventtypekey is NOOP(Non Operational) or its a refund (In this table we can see that refunds are negative in SalesAmount)

## Those those of a null Globalids are: None Reportable SP, NOOP, or a refund(a negative sale or quantity)

## Comparing Current Solution vs New Solution

 first we pulled from the current solution (marketing database) and compared it to the new solution but has been formatted to the current solution.

 For this exersise we used the same order_detail_id for all aspects of different tables

In [ ]:
%%sql -r dataframe_7
SELECT a.crm_customer_no,a.global_id,purchaser_no,order_detail_id, agc_id,b.ext_customer_link_id, b.first_name, b.last_name
FROM SIXFL_PROD.CONSUMPTION.VW_PARK_SALES a 
JOIN SIXFL_PROD.CONSUMPTION.VW_PARK_CUSTOMER b 
    ON a.crm_customer_no = b.crm_customer_no
    AND a.park_id = b.park_id
WHERE 1=1
 AND order_detail_id = '015SW100264348-002' -- looking at comparisons withing different tables
 --AND order_detail_id = '206SW103484634-001' -- looking at comparisons of day tickets where SPkey is like None
 --AND order_detail_id = '305SW101400556-004' -- looking at comparisons of day tickets where ATkey is like None

LIMIT 100

In [ ]:
%%sql -r dataframe_9
SELECT a."SiteKey",a."CRMCustomerNo",a."GlobalID","PurchaserNo", b."FirstName",b."LastName",a."TicketID",a."TicketNo"
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_LEGACY_EXTRACT__SALES a 
JOIN ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_LEGACY_EXTRACT__CUSTOMERS b  
ON a."CRMCustomerNo" = b."CRMCustomerNo" 
WHERE  1=1
AND "OrderDetailId" = '015SW100264348-002'-- looking at comparisons withing different tables
--AND "OrderDetailId" = '206SW103484634-001' -- looking at comparisons of day tickets where spkey is like None


Things we see when comparing with current solution -> new solution but reformatted to match current solution:
1. we see that there is a match in GlobalIds(and park of the CRMCustomerno)
2. In the vw_park_customer we have column EXT_CUSTOMER_LINK_ID which is the same as PurchaseNo  when we remove the "CP|"
3. When we are comparing these two tables, we may not know how the CRM_CUSTOMER_NO is built, but we see that the CRMCustomerNo is a combination of SiteKey + | + GlobalID

Now we need to compare to the reformatted table with the original solution that made this table.

In [ ]:
%%sql -r dataframe_8
SELECT DISTINCT a."SiteKey",a."GlobalId",a."ATCustomerKey", a."SPCustomerKey",b."FirstName" as ATfirstname,c."FirstName" as SPfirstname
FROM ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_FACT_SALES a 
LEFT join ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_ATCUSTOMERS b 
    on a."ATCustomerKey" = b."ATCustomerKey"
LEFT join ORGDATACLOUD$INTERNAL$PROD_CONSUMPTION_CDP_LISTING.CDP.CDP_DIM_SPCUSTOMERS c
    on a."SPCustomerKey" = c."SPCustomerKey"
WHERE 1=1
AND "OrderDetailName" LIKE '015SW100264348-002' -- looking at comparisons withing different tables
--AND "OrderDetailName" LIKE '206SW103484634-001' -- looking at comparisons of day tickets where SPkey is None
--AND "TicketId" LIKE 'CW142694214-153102914' -- looking at comparisons of day tickets where ATkey is None





Things we see when we compare with reformatted solution -> original solution:

1. we see a match in counts of row/GlobalIDs on both tables.
2. **we see that PurchaseNo is matching with SPKey but we learned that SPkey is the owner of the pass and not the purchaser which is ATkey**
3. The individuals that are mentioned in each are match based on the owner of SPkey and not the purchaser. 

Based on this insights we have we can see that we are marketing directly to the owners of the season pass

in the event that there is no SPkey the ATkey would then replace the CRMCustomerNo in the new solution 

we also see that in the current solution those that are with a -1 but have an orderID, are now replaced with a CRMCustomerNo